In [1]:
# Discover repo root and read all CSV files from the per-series folders
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path (BEFORE the import attempt)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    print(f'Added {repo_root} to sys.path')

# Directory containing hourly CSVs
wiertsema_dir = repo_root / 'output_data' / 'csv_wiertsema_validated'
fugro_dir = repo_root / 'output_data' / 'csv_fugro_validated'
out_fig = repo_root / 'output_data' / 'figures'
out_fig.mkdir(parents=True, exist_ok=True)

print('wiertsema_dir ->', wiertsema_dir)
print('fugro_dir    ->', fugro_dir)

Added d:\Users\jvanruitenbeek\data_validation to sys.path
wiertsema_dir -> d:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_validated
fugro_dir    -> d:\Users\jvanruitenbeek\data_validation\output_data\csv_fugro_validated


# Loop over folder and generate report from the filtered files

In [3]:
# Loop over filtered CSV files and generate validation report

import numpy as np  # (optional now, only keep if you use it elsewhere)

# Choose folder: wiertsema or fugro
folder_to_process = wiertsema_dir  # Change to wiertsema_dir if needed

# Get all CSV files
csv_files = sorted(folder_to_process.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files in {folder_to_process.name}\n')

# Initialize list to store metrics
report_data = []

# Loop over each file
for i, csv_file in enumerate(csv_files, start=1):
    try:
        print(f'[{i}/{len(csv_files)}] Processing: {csv_file.name}')
        
        # Read the CSV
        df = pd.read_csv(
            csv_file,
            index_col=0,
            parse_dates=True,
            encoding="utf-8-sig",
            encoding_errors="replace"
        )
        
        # Debugging statement


        # Coerce head column to numeric
        df['head'] = pd.to_numeric(df['head'], errors='coerce')
        head_series = df['head'].dropna()
        
        if len(head_series) == 0:
            print(f'  ✗ No valid head data\n')
            continue

        # Debugging statement
        df.info()
        
        # Calculate metrics
        first_entry = head_series.index[0]
        last_entry = head_series.index[-1]
        num_entries = len(head_series)
        
        # Time span in days
        time_span_days = (last_entry - first_entry).days
        
        # Completeness: entries with values / possible entries (hourly)
        possible_entries = (time_span_days * 24) + 1  # +1 to include both endpoints
        completeness_pct = (num_entries / possible_entries * 100) if possible_entries > 0 else 0
        
        # Largest gap: find the largest time difference between consecutive non-NaN entries
        time_diffs = head_series.index.to_series().diff().dt.total_seconds() / 3600  # Convert to hours
        largest_gap_hours = time_diffs.max() if len(time_diffs) > 1 else 0
        
        # Min and max values
        lowest_value = head_series.min()
        highest_value = head_series.max()
        
        # Largest jump: maximum absolute change between consecutive entries
        head_diff = head_series.diff().abs()
        largest_jump = head_diff.max() if len(head_diff) > 1 else 0
        
        # --- NEW: count how many values were filtered by each algorithm v1–v4 ---
        filter_counts = {}
        for col in ['v1', 'v2', 'v3', 'v4']:
            if col in df.columns:
                # Assumes non-NaN means "this value was filtered/flagged"
                filter_counts[col] = df[col].notna().sum()
            else:
                filter_counts[col] = 0
        
        # Add to report (outliers & jumps/drops removed)
        report_data.append({
            'Filename': csv_file.stem,
            'First Entry': first_entry,
            'Last Entry': last_entry,
            'Number of Entries': num_entries,
            'Completeness (%)': round(completeness_pct, 2),
            'Length (days)': time_span_days,
            'Largest Gap (hours)': round(largest_gap_hours, 2),
            'Lowest Value (m)': round(lowest_value, 4),
            'Highest Value (m)': round(highest_value, 4),
            'Largest Jump (m)': round(largest_jump, 4),
            'Filtered by v1 (count)': filter_counts['v1'],
            'Filtered by v2 (count)': filter_counts['v2'],
            'Filtered by v3 (count)': filter_counts['v3'],
            'Filtered by v4 (count)': filter_counts['v4'],
        })
        
        print(f'  ✓ Metrics calculated\n')
        
    except Exception as e:
        print(f'  ✗ Error processing {csv_file.name}: {e}\n')

# Create DataFrame from report data
report_df = pd.DataFrame(report_data)

# Save to Excel
output_file = repo_root / 'output_data' / f'validation_report_{folder_to_process.name}.xlsx'
report_df.to_excel(output_file, index=False, sheet_name='Validation Report')

print(f'✓ Report saved to: {output_file}')
print(f'\nSummary Statistics:')
print(report_df.describe())

Found 302 CSV files in csv_wiertsema_validated

[1/302] Processing: 83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3795 entries, 2025-06-24 22:00:00 to 2025-11-30 00:00:00
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   head                3753 non-null   float64
 1   head_raw            3762 non-null   float64
 2   Precipitation       3795 non-null   float64
 3   Evapotranspiration  3795 non-null   float64
 4   recharge            3795 non-null   float64
 5   v0                  3795 non-null   bool   
 6   v1                  0 non-null      float64
 7   v2                  1 non-null      float64
 8   v3                  0 non-null      float64
 9   v4                  8 non-null      float64
dtypes: bool(1), float64(9)
memory usage: 300.2 KB
  ✓ Metrics calculated

[2/302] Processing: 83034-1 HB002PB01 BE0049+00_BIKR_GMW_PB1_F-227.csv
  ✗ 